# Gene ↔ Mutation Relation-Wise Merge

Merges Gene–Mutation triples from EvoAGE;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/Mutation_MUTATION/ALL_Mutation_MUTATION.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. hald

In [2]:
hald = pd.read_csv(PROC_DIR + 'hald/Mutation_Mutation_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald: {len(hald):,} rows | columns: {list(hald.columns)}")
hald['kg_type'] = 'Aging'
hald['kg_source'] = 'HALD_KG'
hald['species'] = 'Homo species'

hald.head(2)

hald: 165 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,species
0,rs1805124,Mutation_Mutation,rs7626962,Mutation,associated,Mutation,HALD_KG,Aging,Homo species
1,rs7626962,Mutation_Mutation,rs1805124,Mutation,associated,Mutation,HALD_KG,Aging,Homo species


## 2. Check and Fix Duplicate Columns

In [3]:
SOURCE_DFS = [('hald', hald)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[hald] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [4]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 165 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,rs1805124,Mutation_Mutation,rs7626962,Mutation,associated,Mutation,HALD_KG,<NA>,<NA>,<NA>,<NA>,Aging,Homo species
1,rs7626962,Mutation_Mutation,rs1805124,Mutation,associated,Mutation,HALD_KG,<NA>,<NA>,<NA>,<NA>,Aging,Homo species


## 4. Standardise Fixed-Value Columns

In [5]:
final_df['head_type'] = 'Mutation'
final_df['tail_type'] = 'Mutation'
final_df['relation']  = 'Mutation_Mutation'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Mutation_Mutation'}
Unique head_id_is: {<NA>}
Unique tail_id_is: {<NA>}
Unique kg_source: {'HALD_KG'}


## 5. Deduplicate and Add Schema Columns

In [6]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

After deduplication: 163 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,rs1047776,Mutation_Mutation,rs2238114,Mutation,associated,Mutation,HALD_KG,None,None,None,None,Aging,Homo species
1,rs104893877,Mutation_Mutation,rs104893878,Mutation,expressed,Mutation,HALD_KG,None,None,None,None,Aging,Homo species
2,rs104893877,Mutation_Mutation,rs749476922,Mutation,indicate,Mutation,HALD_KG,None,None,None,None,Aging,Homo species
3,rs104893877,Mutation_Mutation,rs768838511,Mutation,indicate,Mutation,HALD_KG,None,None,None,None,Aging,Homo species
4,rs10505348,Mutation_Mutation,rs2073618,Mutation,associated,Mutation,HALD_KG,None,None,None,None,Aging,Homo species


## 6. QC — NaN Check

In [7]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,163,0,163
tail_id_is,163,0,163
head_detail_name,163,0,163


## 7. Save Output

In [8]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 163 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/Mutation_MUTATION/ALL_Mutation_MUTATION.csv


In [9]:
set(final_df_dedup['kg_type'])

{'Aging'}